In [1]:
from pathlib import Path
import pandas as pd 

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "processed" / "songs_cleaned.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "songs_cleaned.csv"

if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")


songs =pd.read_csv(DATA_PATH)

In [2]:
required_coloums = {"track_id", "track_name", "artists"}
missing =required_coloums-set(songs.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Dataset shape:", songs.shape)
print("Unique track IDs:", songs["track_id"].nunique())

display(songs[["track_id", "track_name", "artists","album_name"]].head())

Dataset shape: (34466, 19)
Unique track IDs: 34466


,track_id,track_name,artists,album_name
0,5SuOikwiRyPMVoIQDJUgSV,Comedy,Gen Hoshino,Comedy
1,4qPNDBW1i3p13qLCt0Ki3A,Ghost - Acoustic,Ben Woodward,Ghost (Acoustic)
2,1iJBSr7s7jYXzM8EGcbK5b,To Begin Again,Ingrid Michaelson;ZAYN,To Begin Again
3,6lfxq3CG4xtTiEg7opyCyx,Can't Help Falling In Love,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...
4,5vjLSffimiIP26QG5WcN2K,Hold On,Chord Overstreet,Hold On


In [3]:
def find_song(song_name: str, songs: pd.DataFrame) -> pd.DataFrame:
    """Find songs with an exact, case-insensitive title match."""

    normalized_query = song_name.strip().casefold()

    normalized_titles = (
        songs["track_name"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.casefold()
    )

    matches = songs.loc[
        normalized_titles == normalized_query,
        ["track_id", "track_name", "artists"]
    ]

    return matches.copy()

In [4]:
matches = find_song("Comedy", songs)

print("Total matches:", len(matches))
display(matches.head(20))

Total matches: 1


,track_id,track_name,artists
0,5SuOikwiRyPMVoIQDJUgSV,Comedy,Gen Hoshino


In [5]:
test_titles = [
    "Comedy",
    "Blinding Lights",
    "Shape of You",
    "ThisSongDoesNotExist123"
]

for title in test_titles:
    matches = find_song(title, songs)

    print(f"{title}: {len(matches)} matching records")

Comedy: 1 matching records
Blinding Lights: 12 matching records
Shape of You: 1 matching records
ThisSongDoesNotExist123: 0 matching records


In [6]:
# A nonexistent song should return no records.
assert find_song("ThisSongDoesNotExist123", songs).empty

# Case differences should not change matching results.
upper_matches = find_song("COMEDY", songs)
lower_matches = find_song("comedy", songs)

assert set(upper_matches["track_id"]) == set(lower_matches["track_id"])

# Every returned track ID should exist in the source dataset.
assert set(upper_matches["track_id"]).issubset(
    set(songs["track_id"])
)

print("Song lookup checks passed!")

Song lookup checks passed!


In [7]:
from typing import TypedDict

class SongState(TypedDict):
    user_input: str
    song: str
    track_id: str
    artist: str
    status: str
    album: str
    response:str
    

In [8]:
def identify_song(state:SongState)-> dict:
    """Extract a song title form a fixed format request"""
    prefix= "Recommend songs like"
    user_input = state["user_input"].strip()

    if not user_input.casefold().startswith(prefix.casefold()):
        return{
            "song": "",
            "status":"invalid_input",
            "response": "Use: Recommend songs like <song title>"
        }

    song_name = user_input[len(prefix):].strip()
    if not song_name:
        return {
            "song": "",
            "status": "invalid_input",
            "response": "Please provide a song title."
        }

    return {
        "song": song_name,
        "status": "pending"
    }

In [9]:
def find_song(song_name: str, songs: pd.DataFrame) -> pd.DataFrame:
    """Find matching songs by title."""

    normalized_query = song_name.strip().casefold()

    normalized_titles = (
        songs["track_name"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.casefold()
    )

    matches = songs.loc[
        normalized_titles == normalized_query,
        ["track_id", "track_name", "artists", "album_name"]
    ]

    return matches.copy()

In [10]:
def lookup_song(state: SongState) -> dict:
    """Look up the identified title in the song dataset."""

    matches = find_song(state["song"], songs)

    if matches.empty:
        return {
            "status": "not_found",
            "response": f"Song '{state['song']}' was not found."
        }

    if len(matches) > 1:
        artists = matches["artists"].dropna().unique().tolist()

        return {
            "status": "ambiguous",
            "response": (
                f"Found {len(matches)} tracks named '{state['song']}'. "
                f"Possible artists: {', '.join(map(str, artists[:10]))}. "
                "Please specify the artist."
            )
        }

    selected = matches.iloc[0]

    return {
        "track_id": str(selected["track_id"]),
        "artist": str(selected["artists"]),
        "status": "found",
        "album":str(selected["album_name"]),
        "response": (
            f"Found '{selected['track_name']}' "
            f"by {selected['artists']}."
            f"from the album'{selected['album_name']}'."
        )
    }

In [11]:
print("album_name" in songs.columns)


True


In [12]:
def route_after_identification(state: SongState) -> str:
    if state["status"] == "invalid_input":
        return "stop"

    return "lookup"


def route_after_lookup(state: SongState) -> str:
    if state["status"] == "found":
        return "success"

    return "stop"

In [13]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(SongState)

builder.add_node("identify_song", identify_song)
builder.add_node("lookup_song", lookup_song)

builder.add_edge(START, "identify_song")

builder.add_conditional_edges(
    "identify_song",
    route_after_identification,
    {
        "lookup": "lookup_song",
        "stop": END
    }
)

builder.add_conditional_edges(
    "lookup_song",
    route_after_lookup,
    {
        "success": END,
        "stop": END
    }
)

graph = builder.compile()

print("LangGraph compiled successfully.")

LangGraph compiled successfully.


In [14]:
test_inputs = [
    "Recommend songs like Comedy",
    "Recommend songs like Blinding Lights",
    "Recommend songs like ThisSongDoesNotExist123",
    "Hello"
]

for user_input in test_inputs:
    initial_state = {
        "user_input": user_input,
        "song": "",
        "track_id": "",
        "artist": "",
        "album": "",
        "status": "",
        "response": ""
    }

    result = graph.invoke(initial_state)

    print("=" * 60)
    print("Input:", user_input)
    print("Status:", result["status"])
    print("Song:", result["song"])
    print("Album:", result["album"])
    print("Track ID:", result["track_id"])
    print("Response:", result["response"])

Input: Recommend songs like Comedy
Status: found
Song: Comedy
Album: Comedy
Track ID: 5SuOikwiRyPMVoIQDJUgSV
Response: Found 'Comedy' by Gen Hoshino.from the album'Comedy'.
Input: Recommend songs like Blinding Lights
Status: ambiguous
Song: Blinding Lights
Album: 
Track ID: 
Response: Found 12 tracks named 'Blinding Lights'. Possible artists: Revelries;Victoria Voss, The Weeknd, All Time Low. Please specify the artist.
Input: Recommend songs like ThisSongDoesNotExist123
Status: not_found
Song: ThisSongDoesNotExist123
Album: 
Track ID: 
Response: Song 'ThisSongDoesNotExist123' was not found.
Input: Hello
Status: invalid_input
Song: 
Album: 
Track ID: 
Response: Use: Recommend songs like <song title>


In [15]:
expected_statuses = [
    "found",
    "ambiguous",
    "not_found",
    "invalid_input"
]

for user_input, expected in zip(test_inputs, expected_statuses):
    result = graph.invoke({
        "user_input": user_input,
        "song": "",
        "track_id": "",
        "artist": "",
        "status": "",
        "response": ""
    })

    assert result["status"] == expected, (
        f"Expected {expected}, got {result['status']}"
    )

print("All LangGraph routing tests passed!")

All LangGraph routing tests passed!


In [16]:
import inspect
import sys

sys.path.insert(0, str(PROJECT_ROOT))

from scripts.recommender import SongRecommender

print("Constructor:")
print(inspect.signature(SongRecommender))

print("\nRecommendation method:")
print(inspect.signature(SongRecommender.recommend))

Constructor:
(data_path: pathlib.Path = WindowsPath('C:/Users/samya/PyCharmProject/song-recommendations-/data/processed/songs_cleaned.csv'), model_dir: pathlib.Path = WindowsPath('C:/Users/samya/PyCharmProject/song-recommendations-/models'))

Recommendation method:
(self, track_id: str, top_n: int = 10) -> pandas.DataFrame


In [17]:
from typing import TypedDict, Any

class SongState(TypedDict):
    user_input: str
    song: str
    track_id: str
    artist: str
    album: str
    status: str
    response: str
    recommendations: list[dict[str, Any]]

In [18]:
SongRecommender()

In [20]:
recommender.recommend(track_id, top_n=10)

NameError: name 'recommender' is not defined